# D09 · CANopen - object dictionary, PDO, SDO, NMT and CiA 402

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D09_canopen_object_dictionary_pdo_sdo_nmt_and_cia_40.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 65 min

**Skill Hub concepts:** CANopen object dictionary; CANopen PDO and SDO; CANopen network management; CiA 402 drive state machine

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D09")

## Map: one node, seen from the bus

| Service | COB-ID | Use |
|---|---|---|
| NMT command | 0x000 | master changes node states: `[cmd, node]` |
| SYNC | 0x080 | triggers synchronous PDOs |
| EMCY | 0x080 + node | error report |
| TPDO1..4 | 0x180 / 0x280 / 0x380 / 0x480 + node | process data out of the node, no protocol overhead |
| RPDO1..4 | 0x200 / 0x300 / 0x400 / 0x500 + node | process data into the node |
| SDO server to client / client to server | 0x580 + node / 0x600 + node | read or write any object dictionary entry, confirmed |
| Heartbeat | 0x700 + node | liveness (lab D08) |

**Object dictionary:** every parameter has a 16-bit index and 8-bit subindex. 0x1000..0x1FFF communication,
0x2000..0x5FFF manufacturer, 0x6000..0x9FFF device profile. CiA 402 drives: 0x6040 controlword, 0x6041 statusword,
0x6060 mode of operation, 0x6064 position actual, 0x607A target position.

**SDO expedited transfer** (all values little-endian, frame always 8 bytes):

| Direction | Byte 0 | Bytes 1..2 | Byte 3 | Bytes 4..7 |
|---|---|---|---|---|
| download (write) request | 0x2F / 0x2B / 0x27 / 0x23 for 1 / 2 / 3 / 4 bytes | index | sub | data |
| download response | 0x60 | index | sub | 0 |
| upload (read) request | 0x40 | index | sub | 0 |
| upload response | 0x4F / 0x4B / 0x47 / 0x43 for 1 / 2 / 3 / 4 bytes | index | sub | data |
| abort | 0x80 | index | sub | 32-bit abort code |

| NMT state | SDO | PDO | Enter with NMT command |
|---|---|---|---|
| Pre-operational | yes | no | 0x80 |
| Operational | yes | yes | 0x01 |
| Stopped | no | no | 0x02 |
| (reset node / reset communication) | | | 0x81 / 0x82 |

**CiA 402 power state machine** (statusword masks):

```
 Not ready ─auto─> Switch on disabled ─0x06 Shutdown─> Ready to switch on ─0x07 Switch on─> Switched on
      (sw & 0x4F == 0x00)   (sw & 0x4F == 0x40)         (sw & 0x6F == 0x21)              (sw & 0x6F == 0x23)
                                    ^                                                          │ 0x0F Enable
  Fault (sw & 0x4F == 0x08) ─0x80 rising edge, fault reset─┘     Operation enabled (sw & 0x6F == 0x27) <┘
  Quick stop active (sw & 0x6F == 0x07)  <─0x02 Quick stop─  Operation enabled;   0x00 Disable voltage -> Switch on disabled
```

## Task 1: SDO frames

In [ ]:
def sdo_download(node, index, sub, value, size):
    """Return (cob_id, bytes of length 8) for an expedited write."""
    raise NotImplementedError  # TODO

def sdo_upload_parse(data):
    """Return (index, sub, value) from an expedited upload response; raise RuntimeError on an abort."""
    raise NotImplementedError  # TODO

lab.check("sdo", sdo_download, sdo_upload_parse)

In [ ]:
%%writefile d09_sdo.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/d09.h
// Grade it with lab.check("sdo", "d09_sdo.c"). For C++, name the file .cpp in both places.
#include "d09.h"

uint16_t sdo_download(uint8_t node, uint16_t index, uint8_t sub, int32_t value, int size, uint8_t *data) {
    (void)node; (void)index; (void)sub; (void)value; (void)size;
    memset(data, 0, 8);
    return 0; /* TODO: command 0x2F/0x2B/0x27/0x23, index little-endian, sub, value little-endian */
}

int sdo_upload_parse(const uint8_t *data, uint16_t *index, uint8_t *sub, uint32_t *value, uint32_t *abort_code) {
    (void)data; *index = 0; *sub = 0; *value = 0; *abort_code = 0;
    return 0; /* TODO: 0x80 = abort; size from bits 2..3 of the command */
}

In [ ]:
# lab.check("sdo", "d09_sdo.c")  # uncomment after writing the C/C++ cell above

## Task 2: bring a drive to "Operation enabled"
`drive.statusword()` reads 0x6041, `drive.write_controlword(cw)` writes 0x6040. Decode the state with the masks,
pick the next controlword, repeat. Some status bits (voltage enabled, remote, warning) vary; the masks ignore them.

In [ ]:
def cia402_state(sw):
    raise NotImplementedError  # TODO

NEXT_CW = {
        "Switch on disabled": None,
        "Ready to switch on": None,
        "Switched on": None,
        "Operation enabled": None,
        "Fault": None,
        "Quick stop active": None,
}

def enable_drive(drive, max_steps=10):
    raise NotImplementedError  # TODO

lab.check("cia402", cia402_state, enable_drive)

In [ ]:
%%writefile d09_cia402.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/d09.h
// Grade it with lab.check("cia402", "d09_cia402.c"). For C++, name the file .cpp in both places.
#include "d09.h"

int cia402_state(uint16_t sw) { (void)sw; return CIA402_UNKNOWN; /* TODO: mask 0x4F or 0x6F and compare */ }

bool enable_drive(const Cia402DriveOps *d, int max_steps) {
    (void)d; (void)max_steps;
    return false; /* TODO: read state, write the next controlword, repeat */
}

In [ ]:
# lab.check("cia402", "d09_cia402.c")  # uncomment after writing the C/C++ cell above

## Task 3: pack a PDO from its mapping

In [ ]:
def pack_pdo(mapping, values):
    """mapping: [(index, sub, bits)], values: {(index, sub): int}. Little-endian, in mapping order."""
    raise NotImplementedError  # TODO

lab.check("pdo", pack_pdo)

In [ ]:
%%writefile d09_pdo.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/d09.h
// Grade it with lab.check("pdo", "d09_pdo.c"). For C++, name the file .cpp in both places.
#include "d09.h"

size_t pack_pdo(const PdoEntry *map, const int64_t *values, size_t n, uint8_t *out) {
    (void)map; (void)values; (void)n; (void)out;
    return 0; /* TODO: each value little-endian, bits/8 bytes, in mapping order */
}

In [ ]:
# lab.check("pdo", "d09_pdo.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** COB-ID of TPDO1 from node 5?

**M2 (Remember).** SDO command byte for an expedited 2-byte download?

**M3 (Apply).** Controlword value for "Enable operation"?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Apply).** The joint's position must reach the controller every 1 ms; its gear ratio is set once at startup.
- a) both over SDO
- b) position in a TPDO (triggered by SYNC), gear ratio written once by SDO
- c) both in PDOs

**K2 (Understand).** Explain NMT states and what traffic each allows, and why a drive must be in "Operational" and
"Operation enabled" before it moves.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d09.h`).
3. In `roboprep/checks/d09.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()